# Stage 2 — Fine-Tune Qwen2.5-7B on Space Flight Data
### Space Flight AI — QLoRA Fine-tuning on H100

This notebook:
1. Loads `finetune_ready.jsonl` generated by `01_pdf_ingestion_pipeline.ipynb`
2. Loads Qwen2.5-7B-Instruct with 4-bit quantization
3. Attaches QLoRA adapters
4. Fine-tunes on space flight decision+reasoning pairs
5. Saves checkpoints and final weights
6. Tests the model's space knowledge and reasoning quality

---
**Requirements:**
- H100 GPU (40GB+ VRAM)
- `finetune_ready.jsonl` from notebook 01
- HuggingFace account with access to Qwen2.5-7B

**Expected training time on H100:** 4-8 hours depending on dataset size

## Step 0 — Install Dependencies

In [ ]:
!pip install -r requirements.txt -q

## Step 1 — Imports and Config

In [ ]:
import os
import json
import torch
import wandb
from pathlib import Path
from dotenv import load_dotenv
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG — edit if needed
# ─────────────────────────────────────────────
BASE_MODEL       = "Qwen/Qwen2.5-7B-Instruct"
DATASET_PATH     = "./finetune_ready.jsonl"
OUTPUT_DIR       = "./checkpoints"
FINAL_MODEL_DIR  = "./space_flight_ai_model"
HF_TOKEN         = os.environ.get("HF_TOKEN")       # set in .env file
WANDB_KEY        = os.environ.get("WANDB_API_KEY")  # set in .env file — optional

# Training hyperparameters
MAX_SEQ_LENGTH   = 2048
BATCH_SIZE       = 8       # H100 can handle this comfortably
GRAD_ACCUM       = 4       # effective batch = 32
EPOCHS           = 3
LEARNING_RATE    = 2e-4
WARMUP_RATIO     = 0.03
VAL_SPLIT        = 0.05    # 5% held out for validation

# QLoRA config
LORA_R           = 64
LORA_ALPHA       = 128
LORA_DROPOUT     = 0.05

# Verify GPU
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Dataset path    : {DATASET_PATH}")
print(f"Output dir      : {OUTPUT_DIR}")

## Step 2 — Load and Inspect Dataset

In [ ]:
# Load JSONL dataset
with open(DATASET_PATH) as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f"Total samples: {len(raw_data)}")
print(f"\nSample entry:")
sample = raw_data[0]
for msg in sample["messages"]:
    print(f"  [{msg['role'].upper()}]")
    print(f"  {msg['content'][:200]}...\n")

# Check dataset is large enough
if len(raw_data) < 200:
    print("WARNING: Dataset is small (<200 samples)")
    print("   Fine-tuning will work but results may be limited.")
    print("   Recommend adding more PDFs and rerunning notebook 01.")
elif len(raw_data) < 1000:
    print("Dataset is moderate (200-1000 samples) — usable but more is better")
else:
    print(f"✓ Dataset size is good ({len(raw_data)} samples)")

## Step 3 — Load Tokenizer with Custom Special Tokens

In [ ]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Logged into HuggingFace")
else:
    print("No HF_TOKEN found — model must already be cached locally")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    trust_remote_code=True
)

# Add domain-specific special tokens
special_tokens = [
    "<|mission_phase|>",
    "<|situation|>",
    "<|reasoning|>",
    "<|theory_ref|>",
    "<|decision|>",
    "<|correction|>",
    "<|telemetry|>"
]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})
tokenizer.padding_side  = "right"   # right padding for SFT training
tokenizer.model_max_length = MAX_SEQ_LENGTH

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded")
print(f"  Vocab size      : {len(tokenizer)}")
print(f"  Special tokens  : {special_tokens}")
print(f"  Max length      : {tokenizer.model_max_length}")

## Step 4 — Load Model with 4-bit Quantization (QLoRA)

In [ ]:
# 4-bit quantization config — fits 7B model in ~8GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",         # NormalFloat4 — best quality
    bnb_4bit_compute_dtype=torch.bfloat16,  # H100 native dtype
    bnb_4bit_use_double_quant=True,    # double quantization saves more memory
)

print(f"Loading {BASE_MODEL}...")
print("This may take 5-10 minutes on first load (downloading ~15GB)")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",                 # automatically uses all available GPUs
    trust_remote_code=True,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)

# Resize embeddings to account for new special tokens
model.resize_token_embeddings(len(tokenizer))

# Disable cache for training
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"✓ Model loaded")
print(f"  Parameters      : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
if torch.cuda.is_available():
    print(f"  VRAM used       : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 5 — Attach QLoRA Adapters

In [ ]:
from peft import prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# QLoRA config — targets all attention projection layers
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention layers
        "gate_proj", "up_proj", "down_proj"         # MLP layers
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Print trainable vs frozen parameter counts
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✓ QLoRA adapters attached")
print(f"  Trainable params : {trainable / 1e6:.1f}M ({100 * trainable / total:.2f}% of total)")
print(f"  Frozen params    : {(total - trainable) / 1e9:.2f}B")
print(f"  LoRA rank        : {LORA_R}")
print(f"  LoRA alpha       : {LORA_ALPHA}")

## Step 6 — Prepare Dataset

In [ ]:
def format_chat(sample):
    """
    Convert messages list into a single training string
    using the model's chat template.
    """
    return tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

# Convert to HuggingFace Dataset
hf_dataset = Dataset.from_list(raw_data)

# Format all samples
hf_dataset = hf_dataset.map(
    lambda x: {"text": format_chat(x)},
    remove_columns=hf_dataset.column_names
)

# Train/validation split
split = hf_dataset.train_test_split(test_size=VAL_SPLIT, seed=42)
train_dataset = split["train"]
val_dataset   = split["test"]

print(f"✓ Dataset prepared")
print(f"  Train samples : {len(train_dataset)}")
print(f"  Val samples   : {len(val_dataset)}")
print(f"\nSample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])

## Step 7 — Configure Training

In [ ]:
# Optional: init Weights & Biases for experiment tracking
if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
    os.environ["WANDB_PROJECT"] = "space-flight-ai"
    report_to = "wandb"
    print("✓ Weights & Biases enabled")
else:
    report_to = "none"
    print("W&B not configured — logging to terminal only")

os.makedirs(OUTPUT_DIR, exist_ok=True)

sft_config = SFTConfig(
    # Output
    output_dir=OUTPUT_DIR,

    # Training duration
    num_train_epochs=EPOCHS,
    max_steps=-1,                      # -1 = use epochs instead

    # Batch size
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Optimizer
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_32bit",         # memory efficient optimizer
    weight_decay=0.001,

    # Precision — H100 native
    fp16=False,
    bf16=True,

    # Sequence
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,                     # set True if dataset is very large

    # Checkpointing
    save_strategy="epoch",
    save_total_limit=3,               # keep last 3 checkpoints
    load_best_model_at_end=True,

    # Evaluation
    eval_strategy="epoch",
    eval_steps=1,

    # Logging
    logging_steps=10,
    report_to=report_to,

    # Misc
    gradient_checkpointing=True,      # saves VRAM at cost of slight slowdown
    group_by_length=True,             # groups similar length samples → faster
    dataloader_num_workers=4,
    seed=42,
)

print("✓ Training config ready")
print(f"  Epochs          : {EPOCHS}")
print(f"  Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Precision       : bfloat16 (H100 native)")
print(f"  Checkpoints → {OUTPUT_DIR}")

## Step 8 — Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

print("Starting training...")
print(f"Estimated time on H100: {len(train_dataset) * EPOCHS / 1000:.1f} - {len(train_dataset) * EPOCHS / 500:.1f} hours")
print("Checkpoints will be saved after each epoch.\n")

train_result = trainer.train()

# Save training metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

print("\n✓ Training complete")
print(f"  Final train loss : {metrics.get('train_loss', 'N/A'):.4f}")
print(f"  Total steps      : {metrics.get('global_step', 'N/A')}")
print(f"  Training time    : {metrics.get('train_runtime', 0) / 3600:.2f} hours")

## Step 9 — Save Final Model and Weights

In [ ]:
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# Save LoRA adapter weights only (~100MB vs 15GB for full model)
trainer.model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Save training config for reproducibility
config_log = {
    "base_model":    BASE_MODEL,
    "dataset_path":  DATASET_PATH,
    "dataset_size":  len(raw_data),
    "epochs":        EPOCHS,
    "batch_size":    BATCH_SIZE,
    "grad_accum":    GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "lora_r":        LORA_R,
    "lora_alpha":    LORA_ALPHA,
    "max_seq_len":   MAX_SEQ_LENGTH,
    "train_loss":    metrics.get("train_loss"),
    "train_runtime_hours": metrics.get("train_runtime", 0) / 3600,
}

with open(f"{FINAL_MODEL_DIR}/training_config.json", "w") as f:
    json.dump(config_log, f, indent=2)

print(f"✓ Model saved → {FINAL_MODEL_DIR}")
print(f"  adapter_model.bin   ← LoRA weights (upload this)")
print(f"  adapter_config.json ← LoRA config")
print(f"  tokenizer files     ← custom tokenizer with special tokens")
print(f"  training_config.json← reproducibility log")

# Show directory size
import subprocess
result = subprocess.run(["du", "-sh", FINAL_MODEL_DIR], capture_output=True, text=True)
print(f"  Total size: {result.stdout.split()[0]}")

## Step 10 — Test Space Knowledge

Load the fine-tuned model and test its understanding of orbital mechanics and flight decisions.

In [ ]:
from transformers import pipeline

# Load fine-tuned model for inference
print("Loading fine-tuned model for testing...")

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)
base.resize_token_embeddings(len(tokenizer))
finetuned_model = PeftModel.from_pretrained(base, FINAL_MODEL_DIR)
finetuned_model.eval()

def ask_model(question: str, max_new_tokens: int = 500) -> str:
    """Ask the fine-tuned model a question and return its response."""
    messages = [
        {"role": "system",  "content": "You are an AI rocket pilot with deep knowledge of orbital mechanics and space flight. Analyse situations and provide detailed reasoning for your decisions."},
        {"role": "user",    "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(finetuned_model.device)

    with torch.no_grad():
        outputs = finetuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    return response

print("✓ Model ready for testing")

## Step 11 — Knowledge Tests

In [ ]:
# Test 1 — Core orbital mechanics knowledge
test_questions = [
    "Explain what a Hohmann transfer is and when you would use it during a mission.",
    "The rocket is at 150km altitude with apoapsis 148km and periapsis 152km. Fuel is at 45%. What is your next action and why?",
    "During ascent at 35km altitude, dynamic pressure has reached 32,000 Pa. What should the pilot do and why?",
    "You need to dock with a space station in a 400km circular orbit. You are currently in a 380km x 400km orbit. Describe your docking approach step by step.",
    "The mission requires a transfer to Mars. What is a launch window and why does it matter for delta-v efficiency?"
]

print("=" * 70)
print("SPACE KNOWLEDGE EVALUATION")
print("=" * 70)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'─' * 70}")
    print(f"TEST {i}: {question}")
    print(f"{'─' * 70}")
    response = ask_model(question)
    print(response)
    print()

## Step 12 — Flight Decision Test

Test the model's ability to produce structured flight decisions in the exact format the RL loop will expect.

In [ ]:
# Simulate a real telemetry situation
flight_scenario = """
<|mission_phase|>Gravity Turn Ascent
<|situation|>altitude: 18500m, velocity: 420 m/s, apoapsis: 52000m, 
periapsis: -6300000m, dynamic_pressure: 31200 Pa, fuel_remaining: 71.3%, 
pitch: 52 degrees east, throttle: 100%
What is the correct decision and reasoning?
"""

print("FLIGHT DECISION TEST")
print("=" * 70)
print("INPUT TELEMETRY:")
print(flight_scenario)
print("\nMODEL RESPONSE:")
print("─" * 70)
response = ask_model(flight_scenario, max_new_tokens=600)
print(response)

# Check if response uses special tokens correctly
print("\n" + "─" * 70)
print("STRUCTURED OUTPUT CHECK:")
for token in ["<|reasoning|>", "<|theory_ref|>", "<|decision|>"]:
    present = token in response
    status = "✓" if present else "✗"
    print(f"  {status} {token} {'found' if present else 'MISSING'}")

## Step 13 — Compare Base vs Fine-Tuned

Same question to base model and fine-tuned model — shows what fine-tuning actually changed.

In [ ]:
# Load base model without LoRA for comparison
base_model_only = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)
base_model_only.resize_token_embeddings(len(tokenizer))
base_model_only.eval()

test_q = "At T+240s, the vehicle has reached 80km altitude with apoapsis at 95km. What maneuver should the pilot perform next and why?"

# Base model response
def ask_base(question):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(base_model_only.device)
    with torch.no_grad():
        outputs = base_model_only.generate(**inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("COMPARISON: BASE vs FINE-TUNED")
print(f"Question: {test_q}")
print("\n" + "=" * 70)
print("BASE MODEL (no fine-tuning):")
print("=" * 70)
print(ask_base(test_q))

print("\n" + "=" * 70)
print("FINE-TUNED MODEL (space knowledge):")
print("=" * 70)
print(ask_model(test_q))

---
## ✓ Fine-tuning Complete

**Output files:**
| File/Folder | Description | Size |
|---|---|---|
| `checkpoints/` | Per-epoch checkpoints | ~300MB each |
| `space_flight_ai_model/` | Final LoRA weights | ~100-200MB |
| `space_flight_ai_model/training_config.json` | Reproducibility log | tiny |

**What to do with the weights:**
- Download `space_flight_ai_model/` folder
- This contains only the LoRA adapter — NOT the full 15GB base model
- To use it: load Qwen2.5-7B base + apply these adapters

**Next step:** `03_rl_training.ipynb` — connect fine-tuned model to KSP simulator and train with PPO